# 06c — ML Pipelines & MLOps Lifecycle

Manual ML is messy: you run cells out of order, forget which preprocessing you applied, and can't reproduce results a week later. Pipelines automate the entire flow — data → preprocess → train → evaluate → deploy — making it reproducible, testable, and deployable.

---
## 1 · sklearn Pipeline Recap

sklearn's `Pipeline` chains preprocessing and model into a single object. `ColumnTransformer` applies different transformations to different column types. Together, they eliminate the most common source of ML bugs: train/test preprocessing mismatch.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, ["age", "income"]),
    ("cat", categorical_transformer, ["city", "gender"]),
])

full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
])

print(full_pipeline)

---
## 2 · End-to-End ML Pipeline

Let's build a complete pipeline on a real-ish dataset: load → preprocess → train → evaluate → save. This is the structure every ML project should follow.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = titanic.frame

df = df[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]].copy()
df["survived"] = df["survived"].astype(int)

X = df.drop("survived", axis=1)
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
X_train.head()

In [ ]:
num_cols = ["age", "fare", "sibsp", "parch"]
cat_cols = ["pclass", "sex", "embarked"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
])

pipeline.fit(X_train, y_train)
print(f"Pipeline fitted ✓")

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

y_pred = pipeline.predict(X_test)
print("\n" + classification_report(y_test, y_pred, target_names=["died", "survived"]))

In [ ]:
import joblib

joblib.dump(pipeline, "titanic_pipeline.pkl")

loaded = joblib.load("titanic_pipeline.pkl")
assert (loaded.predict(X_test) == y_pred).all()
print("Pipeline saved and verified ✓")

### The same pipeline as a standalone script

In production, you don't run notebooks. You run scripts. Here's the same pipeline as `train.py`:

In [ ]:
%%writefile train.py
"""Train a Titanic survival classifier and save the pipeline."""
import joblib
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

NUM_COLS = ["age", "fare", "sibsp", "parch"]
CAT_COLS = ["pclass", "sex", "embarked"]


def load_data():
    titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
    df = titanic.frame[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]].copy()
    df["survived"] = df["survived"].astype(int)
    X = df.drop("survived", axis=1)
    y = df["survived"]
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


def build_pipeline():
    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUM_COLS),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), CAT_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
    ])


def main():
    X_train, X_test, y_train, y_test = load_data()
    pipeline = build_pipeline()

    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
    print(f"CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    pipeline.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, pipeline.predict(X_test))
    print(f"Test accuracy: {test_acc:.3f}")

    joblib.dump(pipeline, "titanic_pipeline.pkl")
    print("Model saved to titanic_pipeline.pkl")


if __name__ == "__main__":
    main()

```bash
python train.py
```

---
## 3 · CI/CD for ML

CI/CD (Continuous Integration / Continuous Deployment) automates testing and deployment when you push code. For ML, this means:

1. **Push code** → CI runs tests on data quality and model performance
2. **Tests pass** → automatically retrain on latest data
3. **Model meets threshold** → deploy to production

No manual steps. No "I forgot to retrain after the data update."

### Automated testing for ML

ML tests are different from regular software tests. You're testing data quality, model performance, and pipeline integrity.

In [ ]:
%%writefile test_pipeline.py
"""Tests for the ML pipeline — run with pytest."""
import joblib
import numpy as np
import pandas as pd
import pytest
from sklearn.datasets import fetch_openml


def get_test_data():
    titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
    df = titanic.frame[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]].copy()
    return df


def test_data_not_empty():
    df = get_test_data()
    assert len(df) > 0, "Dataset is empty"


def test_no_duplicate_rows():
    df = get_test_data()
    dup_ratio = df.duplicated().mean()
    assert dup_ratio < 0.05, f"Too many duplicates: {dup_ratio:.1%}"


def test_target_distribution():
    df = get_test_data()
    survived_ratio = df["survived"].astype(int).mean()
    assert 0.2 < survived_ratio < 0.8, f"Extreme class imbalance: {survived_ratio:.2f}"


def test_model_accuracy_above_threshold():
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    df = get_test_data()
    df["survived"] = df["survived"].astype(int)
    X = df.drop("survived", axis=1)
    y = df["survived"]
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    model = joblib.load("titanic_pipeline.pkl")
    acc = accuracy_score(y_test, model.predict(X_test))
    assert acc > 0.75, f"Accuracy {acc:.3f} below threshold 0.75"


def test_model_handles_missing_values():
    model = joblib.load("titanic_pipeline.pkl")
    row = pd.DataFrame([{
        "pclass": 1, "sex": "female", "age": np.nan,
        "sibsp": 0, "parch": 0, "fare": 50.0, "embarked": np.nan,
    }])
    pred = model.predict(row)
    assert pred[0] in [0, 1], "Prediction should be 0 or 1"

```bash
pytest test_pipeline.py -v
```

### GitHub Actions workflow

This YAML file goes in `.github/workflows/ml_pipeline.yml`. Every push triggers: install → test → train → evaluate.

In [ ]:
gh_actions_yaml = """
name: ML Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test-and-train:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run data quality tests
        run: pytest test_pipeline.py -v -k "not model_accuracy"

      - name: Train model
        run: python train.py

      - name: Run model performance tests
        run: pytest test_pipeline.py -v -k "model_accuracy"

      - name: Upload model artifact
        uses: actions/upload-artifact@v4
        with:
          name: trained-model
          path: titanic_pipeline.pkl
"""
print(gh_actions_yaml.strip())

---
## 4 · Data Versioning with DVC

Git tracks code, but it can't handle large data files (CSV dumps, image datasets, model weights). DVC (Data Version Control) fills this gap — it works alongside git to version your data.

```bash
pip install dvc
dvc init               # initialize in a git repo
dvc add data/train.csv # track the file (creates .dvc pointer)
git add data/train.csv.dvc .gitignore
git commit -m "Track training data"
```

DVC stores the actual data in a remote (S3, GCS, local folder) and keeps a lightweight pointer in git. When a teammate clones the repo:

```bash
dvc pull   # downloads the exact data version for this commit
```

Now you can go back to any git commit and get the exact data that was used for that experiment.

In [ ]:
dvc_example = """
# Typical DVC workflow

dvc init
dvc remote add -d storage s3://my-bucket/dvc-store

# Track data
dvc add data/raw/customers.csv
git add data/raw/customers.csv.dvc data/raw/.gitignore
git commit -m "Add raw customer data v1"

# Push data to remote storage
dvc push

# Later — update data and version it
dvc add data/raw/customers.csv
git add data/raw/customers.csv.dvc
git commit -m "Update customer data v2"
dvc push

# Go back to the old version
git checkout HEAD~1 -- data/raw/customers.csv.dvc
dvc checkout
"""
print(dvc_example.strip())

---
## 5 · Model Monitoring

Deploying a model isn't the end — it's the beginning of a new set of problems. Models degrade over time because the world changes.

**Data drift**: the input distribution shifts. Your model was trained on summer data, but now it's winter and user behavior is different.

**Concept drift**: the relationship between inputs and outputs changes. What used to predict churn no longer does because the product changed.

**How to detect it**: track prediction distributions, feature distributions, and (when you have labels) actual model accuracy over time.

In [ ]:
import numpy as np
from scipy import stats

train_ages = np.random.normal(35, 10, 1000)
recent_ages = np.random.normal(42, 12, 500)

ks_stat, p_value = stats.ks_2samp(train_ages, recent_ages)
print(f"KS statistic: {ks_stat:.3f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("⚠️  DATA DRIFT DETECTED — distribution has shifted significantly")
else:
    print("✓ No significant drift")

In [ ]:
def check_model_health(weekly_accuracies, threshold=0.75, decline_threshold=0.05):
    """Alert if model performance drops below threshold or declines over time."""
    alerts = []

    latest = weekly_accuracies[-1]
    if latest < threshold:
        alerts.append(f"Accuracy {latest:.3f} below threshold {threshold}")

    if len(weekly_accuracies) >= 4:
        recent_avg = np.mean(weekly_accuracies[-4:])
        older_avg = np.mean(weekly_accuracies[:4])
        decline = older_avg - recent_avg
        if decline > decline_threshold:
            alerts.append(f"Accuracy declining: {older_avg:.3f} → {recent_avg:.3f} (Δ={decline:.3f})")

    return alerts

weekly_acc = [0.91, 0.90, 0.89, 0.88, 0.85, 0.83, 0.80, 0.76]
alerts = check_model_health(weekly_acc)
for alert in alerts:
    print(f"🚨 {alert}")

---
## 6 · The Full MLOps Lifecycle

MLOps is not a one-time process — it's a **loop**. You deploy a model, monitor it, detect drift, retrain on new data, and deploy again.

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│   Data ──→ Preprocess ──→ Train ──→ Evaluate        │
│     ↑                                    │          │
│     │                                    ↓          │
│  Retrain ←── Monitor ←── Deploy ←── Register        │
│                                                     │
└─────────────────────────────────────────────────────┘
```

| Stage | Tools | What Happens |
|---|---|---|
| **Data** | DVC, S3, databases | Collect and version datasets |
| **Preprocess** | sklearn Pipeline, pandas | Clean, transform, feature engineer |
| **Train** | sklearn, PyTorch, XGBoost | Fit model on training data |
| **Evaluate** | pytest, cross-validation | Verify model meets performance bar |
| **Register** | MLflow Model Registry | Version and stage the model |
| **Deploy** | Docker, FastAPI, Cloud Run | Serve predictions via API |
| **Monitor** | custom scripts, Evidently | Track drift and performance |
| **Retrain** | Airflow, GitHub Actions | Triggered by drift or schedule |

In [ ]:
lifecycle = {
    "stages": [
        {"name": "Data Collection",   "trigger": "new data arrives"},
        {"name": "Preprocessing",     "trigger": "pipeline start"},
        {"name": "Training",          "trigger": "preprocessed data ready"},
        {"name": "Evaluation",        "trigger": "training complete"},
        {"name": "Registration",      "trigger": "eval passes threshold"},
        {"name": "Deployment",        "trigger": "new model registered"},
        {"name": "Monitoring",        "trigger": "model serving traffic"},
        {"name": "Retraining",        "trigger": "drift detected or scheduled"},
    ]
}

for i, stage in enumerate(lifecycle["stages"]):
    arrow = "  →  " if i < len(lifecycle["stages"]) - 1 else "  ↩  (back to Data Collection)"
    print(f"{i+1}. {stage['name']:20s} triggered by: {stage['trigger']}{arrow if i == len(lifecycle['stages'])-1 else ''}")

---
## Summary

| Concept | Key Idea |
|---|---|
| sklearn Pipeline | Chain preprocessing + model → single `.fit()` / `.predict()` |
| Script over notebook | Production ML runs as `python train.py`, not cell-by-cell |
| ML testing | Test data quality, model performance, edge cases |
| CI/CD | GitHub Actions automates test → train → deploy on every push |
| DVC | Version large data files alongside git |
| Monitoring | Detect data drift and concept drift before users notice |
| MLOps lifecycle | It's a loop: train → deploy → monitor → retrain |